# Safety Jacket Detection - Auto-Annotation & Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anzonathan/Traffic-Police-Agent/blob/main/notebooks/train_jacket_model_colab.ipynb)

> ⚠️ **Before running:** Set runtime to GPU — `Runtime → Change runtime type → T4 GPU`

**Dataset:** `UCUResearchLab/Reflector-Jackets` — 301 unannotated images.

**Pipeline:**
1. Install dependencies
2. Download dataset from HuggingFace
3. Save images with 80/20 train/val split
4. Auto-annotate with **Grounding DINO**
5. Train **YOLOv8**
6. Download `best.pt` to your local machine

## Step 1: Verify GPU & Install Dependencies

In [1]:
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

GPU ready: Tesla T4


In [2]:
!pip install -q datasets transformers ultralytics pillow pyyaml accelerate
print("Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00
Dependencies installed!


## Step 2: Download Dataset from HuggingFace

In [3]:
import os, random
from datasets import load_dataset
from PIL import Image

DATASET_NAME = "UCUResearchLab/Reflector-Jackets"
print(f"Loading '{DATASET_NAME}'...")

ds = load_dataset(DATASET_NAME)
print(f"Dataset: {ds}")

# Auto-detect the image column
sample = ds['train'][0]
image_col = next(
    (k for k, v in sample.items() if hasattr(v, 'mode') and hasattr(v, 'save')),
    None
)
assert image_col, "No image column found. Inspect ds['train'].features above."
print(f"Image column: '{image_col}' | Total: {len(ds['train'])} images")

Loading 'UCUResearchLab/Reflector-Jackets'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/301 [00:00<?, ?it/s]

20260225_075606.jpg:   0%|          | 0.00/2.52M [00:00<?, ?B/s]

20260225_075602(0).jpg:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

PXL_20260225_120700711.jpg:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

PXL_20260225_120638693.MP.jpg:   0%|          | 0.00/6.28M [00:00<?, ?B/s]

PXL_20260225_120653507.jpg:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/301 [00:00<?, ? examples/s]

Dataset: DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 301
    })
})
Image column: 'image' | Total: 301 images


## Step 3: Save Images with 80/20 Train/Val Split

In [4]:
for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

all_idx = list(range(len(ds['train'])))
random.seed(42)
random.shuffle(all_idx)

train_set = set(all_idx[:int(len(all_idx) * 0.8)])

for idx, item in enumerate(ds['train']):
    split_name = "train" if idx in train_set else "val"
    img = item[image_col]
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(f"dataset/images/{split_name}/img_{idx}.jpg")

print(f"Train: {len(train_set)} | Val: {len(all_idx) - len(train_set)} images saved.")

Train: 240 | Val: 61 images saved.


## Step 4: Auto-Annotate with Grounding DINO

> ⏳ Slowest step — ~10-20 min on T4 GPU.

In [ ]:
import glob
from transformers import pipeline

device = "cuda"
print("Loading Grounding DINO...")
dino = pipeline(
    task="zero-shot-object-detection",
    model="IDEA-Research/grounding-dino-tiny",
    device=device
)

LABELS = ["reflector jacket", "safety vest", "high visibility jacket", "orange safety jacket"]
CONF = 0.25

def annotate_split(split_name):
    paths = sorted(glob.glob(f"dataset/images/{split_name}/*.jpg"))
    annotated = 0
    for i, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        dets = dino(img, candidate_labels=LABELS)
        label_path = f"dataset/labels/{split_name}/{os.path.basename(path).replace('.jpg', '.txt')}"
        lines = []
        for d in dets:
            if d["score"] < CONF:
                continue
            b = d["box"]
            xc = ((b["xmin"] + b["xmax"]) / 2) / w
            yc = ((b["ymin"] + b["ymax"]) / 2) / h
            bw = (b["xmax"] - b["xmin"]) / w
            bh = (b["ymax"] - b["ymin"]) / h
            lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        with open(label_path, "w") as f:
            f.write("\n".join(lines))
        annotated += bool(lines)
        if (i + 1) % 25 == 0:
            print(f"  [{split_name}] {i+1}/{len(paths)}...")
    print(f"  [{split_name}] Done — {annotated}/{len(paths)} images annotated.")

annotate_split("train")
annotate_split("val")
print("Annotation complete!")

Loading Grounding DINO...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [train] 25/240...
  [train] 50/240...
  [train] 75/240...
  [train] 100/240...
  [train] 125/240...
  [train] 150/240...
  [train] 175/240...
  [train] 200/240...
  [train] 225/240...
  [train] Done — 167/240 images annotated.
  [val] 25/61...


## Step 5: Train YOLOv8

In [ ]:
import yaml
from ultralytics import YOLO

dataset_yaml = {
    "path": os.path.abspath("dataset"),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "safety_jacket"}
}
with open("dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

model = YOLO("yolov8n.pt")
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,       # T4 can handle larger batches
    device="cuda",
    patience=20,
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/detect",
    name="safety_jacket_v1"
)
print("Training complete!")

## Step 6: Evaluate & Download `best.pt`

After downloading, place it in your local project at `models/best_jacket.pt`.

In [ ]:
from google.colab import files

best_model_path = "runs/detect/safety_jacket_v1/weights/best.pt"

if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(data="dataset.yaml")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    print("\nDownloading best.pt...")
    files.download(best_model_path)
else:
    print(f"Not found: {best_model_path}. Check the runs/ directory.")